# Macroinvertebrates Database Schema

In [1]:
import pandas as pd

In [2]:
# Config cell
DB_PATH = "macroinvertebrates.db"

In [3]:
# SET UP 
import sqlite3

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

def run_sql(sql: str):
    try:
        cur = conn.execute(sql)
        if sql.lstrip().upper().startswith(("SELECT", "PRAGMA")):
            rows = cur.fetchall()
            print(rows)
        else:
            conn.commit()
            print("OK")
    except sqlite3.Error as err:
        print(f"SQLite error: {err}")

In [ ]:

def create_tables(conn):
    
    cur = conn.cursor()
    cur.executescript("""
    DROP TABLE IF EXISTS Year;
    DROP TABLE IF EXISTS Date;
    DROP TABLE IF EXISTS Sample;
    DROP TABLE IF EXISTS Macroinvertebrate;
    DROP TABLE IF EXISTS Taxonomy;
    """)


    cur.executescript("""

    CREATE TABLE Year (
        year INT CHECK(year >= 2019),
        winterSediment REAL,
        PRIMARY KEY(year)
    );
        
    CREATE TABLE Date (
        date VARCHAR(5),   
        year INT,
        monthlyAveragePrecipitation REAL,
        maxDischarge REAL,
        medianDischarge REAL,
        aggregatedDegreeDays REAL,
        maxTurb REAL,
        medianTurb REAL,
        maxWaterTemp REAL,
        medWaterTemp REAL,
        season TEXT CHECK(season=='Spring' OR season=='Fall' OR season=='Summer'),
        PRIMARY KEY(date),
        FOREIGN KEY(year) REFERENCES Year(year)
    );

        
    CREATE TABLE Sample (
        sampleID VARCHAR(12),  
        date VARCHAR(5), 
        year INT,
        startTime TEXT, 
        sampleMethod TEXT,
        quadrat REAL,
        location TEXT CHECK(location=='Upstream' OR location =='Downstream'),
        microhabitat TEXT CHECK(microhabitat == 'DSR' OR microhabitat == 'DSP' OR microhabitat == 'DFR' OR microhabitat == 'DM' OR microhabitat ==  'DSH' OR microhabitat == 'USR' OR microhabitat == 'UFR' OR microhabitat == 'UM' OR  microhabitat == 'USH' OR microhabitat == 'USU' OR microhabitat == NULL),
        waterDepth REAL,
        percentSediment REAL,
        percentRock REAL CHECK (percentRock >= 0 AND percentRock <= 100),
        percentOrganic REAL CHECK (percentOrganic>=0 AND percentOrganic <= 100),
        pH REAL CHECK (pH >= 0 AND pH <= 14),
        waterTemp REAL,
        dissolvedO2 REAL,
        light REAL,
        flow REAL,
        turb REAL, 
        conductivity REAL, 
        nitrate REAL,
        alkalinity INT,
        PRIMARY KEY(sampleID, startTime, quadrat),
        FOREIGN KEY(date) REFERENCES Date(date)
    );

    CREATE TABLE Macroinvertebrate (
        scientificName TEXT,  
        sampleID VARCHAR(12),
        quadrat REAL,
        stage REAL CHECK(stage == 'larva' OR stage == 'pupa' OR stage == 'adult'),
        numOrganismsFound INT CHECK (numOrganismsFound >= 0),
        benthicArea REAL,
        invertebrateDensity REAL,
        PRIMARY KEY(scientificName, sampleID, quadrat),
        FOREIGN KEY(sampleID) REFERENCES Sample(sampleID)
        FOREIGN KEY(quadrat) REFERENCES Sample(quadrat)
    );

    CREATE TABLE Taxonomy (
        taxonID VARCHAR(6) UNIQUE,      
        scientificName TEXT UNIQUE,
        taxonRank TEXT,
        taxgroup TEXT,
        phylum TEXT,
        subphylum TEXT,
        class TEXT,
        taxOrder TEXT,
        family TEXT,
        subfamily TEXT,
        tribe TEXT,
        genus TEXT,
        taxaNotes TEXT, 
        tolerance INT,
        ffg VARCHAR(3) CHECK(ffg == 'cf' OR ffg == 'cg' or ffg =='om' OR ffg == 'prc' OR ffg == 'prd' OR ffg == 'scr' OR ffg == 'sh'), 
        ffg2 VARCHAR(3) CHECK(ffg2 == 'cf' OR ffg2 == 'cg' or ffg2 =='om' OR ffg2 == 'prc' OR ffg2 == 'prd' OR ffg2 == 'scr' OR ffg2 == 'sh' OR ffg2 == NULL),
        PRIMARY KEY(taxonID),
        FOREIGN KEY(scientificName) REFERENCES Macroinvertebrate(scientificName)
    );
 """)

    conn.commit()

def main():
    conn = sqlite3.connect(DB_PATH)
    create_tables(conn)
    cur = conn.cursor()

    # READ IN ALL DATABASES
    
    env_df = pd.read_csv("env.csv")
    env_df = pd.DataFrame(env_df)

    env_df.rename(columns = {
        'mon.precip' : 'monthlyAveragePrecipitation',
        'mon.ADD' : 'aggregatedDegreeDays',
        'mon.max.discharge' : 'maxDischarge',
        'mon.median.discharge' : 'medianDischarge',
        'mon.max.turb' : 'maxTurb',
        'mon.median.turb' : 'medianTurb',
        'mon.max.wTemp' : 'maxWaterTemp',
        'mon.median.wTemp' : 'medWaterTemp',
        'per_sediment' : 'percentSediment',
        'winter.sediment' : 'winterSediment',
        'per_rock' : 'percentRock',
        'per_organic' : 'percentOrganic',
        'wTemp': 'waterTemp',
        'DO' : 'dissolvedO2',
        'cond' : 'conductivity',
        'depth' : 'waterDepth'
    }, inplace = True)

    macro_df = pd.read_csv("macros.csv")
    macro_df = pd.DataFrame(macro_df)

    macro_df.rename(columns = {
        'number' :'numOrganismsFound',
        'invDens' : 'invertebrateDensity'
    }, inplace = True)

    taxa_df = pd.read_csv("master.taxa.csv")
    taxa_df = pd.DataFrame(taxa_df)

    taxa_df.rename(columns = {
        'taxon_group' :'taxgroup',
        'acceptedTaxonID' : 'taxonID',
        'taxa.notes' : 'taxaNotes',
        'order' : 'taxOrder',
        'FFG' : 'ffg',
        'FFG2' : 'ffg2'
    }, inplace = True)

    water_quality_df = pd.read_csv("waterQ.csv")
    water_quality_df = pd.DataFrame(water_quality_df)
    
    water_quality_df.rename(columns = {
        'depth' : 'waterDepth',
        'wTemp' : 'waterTemp',
        'DO': 'dissolvedO2',
        'cond': 'conductivity'
    }, inplace = True)

    # MAKE YEAR SUBTABLE
    year_subtable = env_df[['year','winterSediment']]
    #print(year_subtable)

    # MAKE TAXA SUBTABLE
    taxa_subtable = taxa_df[['taxonID', 'scientificName', 'taxonRank','taxgroup','phylum', 'subphylum','class','taxOrder', 'family','subfamily','tribe', 'genus','taxaNotes', 'tolerance', 'ffg', 'ffg2']]

    # MAKE DATE SUBTABLE
    date_subtable = env_df[['date','year','monthlyAveragePrecipitation','maxDischarge','medianDischarge','aggregatedDegreeDays','maxTurb','medianTurb','maxWaterTemp', 'medWaterTemp','season']]

    # Join Water Quality and Env
    merged_df = water_quality_df.join(env_df.set_index('sampleID'), on='sampleID', how='outer', lsuffix='_wq', rsuffix='_env')
    merged_df.reset_index(inplace = True)

    # MAKE SAMPLE SUBTABLE
    sample_subtable = merged_df[['sampleID', 'year_wq', 'startTime', 'location_wq','microhabitat_wq','waterDepth_wq', 'percentSediment','percentRock','percentOrganic','pH_wq','waterTemp_wq','dissolvedO2_wq', 'light_wq', 'flow_wq', 'turb_wq','conductivity_wq','nitrate_wq','alkalinity_wq', 'quadrat']]
    # print(sample_subtable)

    # MAKE MACROS SUBTABLE 
    macros_subtable = macro_df[['scientificName', 'sampleID', 'stage', 'numOrganismsFound', 'benthicArea','invertebrateDensity']]

    print(year_subtable.groupby('year'))
    # print(year_subtable)
    # INSERT CSV DATA INTO SQL TABLES

    # cur.executemany(
    #     'INSERT INTO "Year" ("year", "winterSediment")', year_subtable
    # )
    # print(macros_subtable['sampleID'].unique())
    # print(macros_subtable.duplicated(subset=['scientificName']))
    # print(macros_subtable.duplicated(subset=['sampleID']))
    cur.executemany(
            'INSERT INTO "Macroinvertebrate" ("scientificName", "sampleID","stage", "numOrganismsFound","benthicArea","invertebrateDensity") VALUES (?, ?, ?, ?, ?, ?);', macros_subtable.values.tolist()
    )

    cur.executemany(
            'INSERT INTO "Taxonomy" ("taxonID", "scientificName", "taxonRank","taxgroup","phylum", "subphylum","class","taxOrder", "family","subfamily","tribe", "genus","taxaNotes", "tolerance", "ffg", "ffg2") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);', taxa_subtable.values.tolist()
    )
    cur.executemany(
            'INSERT INTO "Sample" ("sampleID", "startTime", "year", "location","microhabitat","waterDepth", "percentSediment","percentRock","percentOrganic","pH","waterTemp","dissolvedO2", "light", "flow", "turb","conductivity","nitrate","alkalinity", "quadrat") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);', sample_subtable.values.tolist()
    )
    #cur.executemany(
    #        'INSERT INTO "Date" ("date","year","monthlyAveragePrecipitation","maxDischarge","medianDischarge","aggregatedDegreeDays","maxTurb","medianTurb","maxWaterTemp", "medWaterTemp","season") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);', date_subtable.values.tolist()
    #)
    # cur.executemany(
    #         'INSERT INTO "Year" ("year", "winterSediment") VALUES (?, ?);', year_subtable.values.tolist()
    # )

    conn.commit()
    
if __name__ == "__main__":
    main();


     year  winterSediment
0    2019               0
1    2019               0
2    2019               0
3    2019               0
4    2019               0
..    ...             ...
330  2024           10300
331  2024           10300
332  2024           10300
333  2024           10300
334  2024           10300

[335 rows x 2 columns]


# Insert Test

In [5]:
run_sql(""" 
    INSERT INTO Year
    VALUES (1978, 10); 
""")

SQLite error: CHECK constraint failed: year >= 2019


In [6]:
run_sql(""" 
    SELECT * 
    FROM Macroinvertebrate
""")

[('Bothrioneurum vejdovskyanum', '01DRSU19.DSP', None, 'adult', 1, 0.5, 2.0), ('Caecidotea sp.', '01DRSU19.DSR', None, 'adult', 1, 0.5, 2.0), ('Ferrissia sp.', '01DRSU19.DM', None, 'adult', 1, 0.5, 2.0), ('Girardia sp.', '01DRSU19.DM', None, 'adult', 1, 0.5, 2.0), ('Girardia sp.', '01DRSU19.DSH', None, 'adult', 4, 0.5, 8.0), ('Girardia sp.', '01DRSU19.DSP', None, 'adult', 3, 0.5, 6.0), ('Lebertia sp.', '01DRSU19.DM', None, 'adult', 2, 0.5, 4.0), ('Lebertia sp.', '01DRSU19.DSH', None, 'adult', 2, 0.5, 4.0), ('Lebertia sp.', '01DRSU19.DSP', None, 'adult', 1, 0.5, 2.0), ('Lebertia sp.', '01DRSU19.DSR', None, 'adult', 2, 0.5, 4.0), ('Lumbriculus variegatus', '01DRSU19.DSH', None, 'adult', 2, 0.5, 4.0), ('Optioservus sp.', '01DRSU19.DFR', None, 'adult', 1, 0.5, 2.0), ('Optioservus sp.', '01DRSU19.DM', None, 'adult', 3, 0.5, 6.0), ('Optioservus sp.', '01DRSU19.DSH', None, 'adult', 2, 0.5, 4.0), ('Optioservus sp.', '01DRSU19.DSP', None, 'adult', 2, 0.5, 4.0), ('Oulimnius sp.', '01DRSU19.DFR',

In [7]:
run_sql(""" 
    SELECT * 
    FROM Taxonomy
""")

[('AEOSP2', 'Aeolosoma sp.', 'genus', 'segmentedWorms', 'Annelida', None, 'Clitellata', 'Aeolosomatida', 'Aeolosomatidae', None, None, 'Aeolosoma', None, 8, 'cf', None), ('BRASP9', 'Branchiobdellidae sp.', 'family', 'leeches', 'Annelida', None, 'Clitellata', 'Branchiobdellida', 'Branchiobdellidae', None, None, None, None, 6, 'cg', None), ('ENCSP1', 'Enchytraeidae sp.', 'family', 'segmentedWorms', 'Annelida', None, 'Clitellata', 'Enchytraeida', 'Enchytraeidae', None, None, None, None, 10, 'cg', None), ('HAPSP4', 'Haplotaxis sp.', 'genus', 'segmentedWorms', 'Annelida', None, 'Clitellata', 'Haplotaxida', 'Haplotaxidae', None, None, 'Haplotaxis', None, 5, 'prd', None), ('ERPSP3', 'Erpobdella sp.', 'genus', 'leeches', 'Annelida', None, 'Clitellata', 'Hirudina', 'Erpobdellidae', 'Erpobdellinae', None, 'Erpobdella', None, 8, 'prd', None), ('GLOCOM', 'Glossiphonia complanata', 'species', 'leeches', 'Annelida', None, 'Clitellata', 'Hirudina', 'Glossiphoniidae', 'Glossiphoniinae', None, 'Glossip

In [8]:
run_sql(""" 
    SELECT * 
    FROM Sample
""")

[('01DRFA19.DFR', None, '14:00:00', '2019', None, 1.0, 'Downstream', 'DFR', 19.0, 47.5, 45.0, 7.5, 7.48, 19.1, 9.83, None, None, 12.0, 204.9, None, None), ('01DRFA19.DFR', None, '14:20:00', '2019', None, 2.0, 'Downstream', 'DFR', 15.0, 47.5, 45.0, 7.5, None, None, None, None, None, 8.4, 205.0, None, None), ('01DRFA19.DFR', None, None, '2019', None, 3.0, 'Downstream', 'DFR', None, 47.5, 45.0, 7.5, None, None, None, None, None, 11.4, 203.9, None, None), ('01DRFA19.DM', None, '13:49:00', '2019', None, 1.0, 'Downstream', 'DM', 22.5, 7.5, 91.0, 1.5, 7.48, 19.2, 9.91, None, None, 7.6, 205.3, None, None), ('01DRFA19.DM', None, '14:30:00', '2019', None, 2.0, 'Downstream', 'DM', 18.0, 7.5, 91.0, 1.5, None, None, None, None, None, 23.6, 205.1, None, None), ('01DRFA19.DM', None, None, '2019', None, 3.0, 'Downstream', 'DM', None, 7.5, 91.0, 1.5, None, None, None, None, None, 11.1, 223.7, None, None), ('01DRFA19.DSH', None, '13:49:00', '2019', None, 1.0, 'Downstream', 'DSH', 17.0, 22.5, 75.0, 2.5, 

In [9]:
run_sql(""" 
    DELETE
    FROM Year
""")

OK
